In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, col, year
from pyspark.sql.functions import split, explode
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import when

In [0]:
#Chargement du fichier Steam
filepath = 's3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json'

In [0]:
#Afficher le schéma de la base de données et 5 lignes d'examples

steam = (spark.read.format('json')\
             .option('header', 'true')\
             .option('inferSchema', 'true')\
             .load(filepath))
steam.show(5)

+--------------------+-------+
|                data|     id|
+--------------------+-------+
|{10, [Multi-playe...|     10|
|{1000000, [Single...|1000000|
|{1000010, [Single...|1000010|
|{1000030, [Multi-...|1000030|
|{1000040, [Single...|1000040|
+--------------------+-------+
only showing top 5 rows



In [0]:
#Une autre façon d'afficher le schéma de la base de données
steam.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
#nombre de lignes 
steam.count()


Out[13]: 55691

In [0]:
#nombre de colonnes même les colonnes imbriquées
from pyspark.sql.types import StructType

def count_all_columns(schema):
    count = 0
    for field in schema.fields:
        if isinstance(field.dataType, StructType):
            count += count_all_columns(field.dataType)
        else:
            count += 1
    return count

# Utilisation
nb_total_colonnes = count_all_columns(steam.schema)
print(f"Nombre total de colonnes (y compris imbriquées) : {nb_total_colonnes}")


Nombre total de colonnes (y compris imbriquées) : 465


In [0]:
nlines_avant = steam.count()
nlines_dupliques = steam.dropDuplicates().count()
print("Duplicates Rows :", nlines_avant != nlines_dupliques)

Duplicates Rows : False


In [0]:
# 1. Compter les lignes totales
total_count = steam.count()
print(f"Nombre total de lignes : {total_count}")

# 2. Compter les valeurs manquantes pour la colonne struct 'data' elle-même
missing_data_count = steam.filter(F.col("data").isNull()).count()
print(f"Nombre de lignes où 'data' est null : {missing_data_count}")

# 3. Compter les valeurs manquantes pour chaque sous-colonne de 'data'
# On sélectionne toutes les sous-colonnes de 'data'
data_fields = steam.schema["data"].dataType.fieldNames() # Liste des noms de champs sous 'data'


missing_counts = []

# Pour chaque champ sous 'data', on compte les nulls
for field in data_fields:
    null_count = steam.filter(F.col(f"data.{field}").isNull()).count()
    missing_counts.append((field, null_count))

# Création d'un DataFrame à partir de la liste
missing_df = spark.createDataFrame(missing_counts, ["Column", "Missing_Count"])

# Ajout d'une colonne pour le pourcentage
missing_df_with_percent = missing_df.withColumn(
    "Percentage", 
    (F.col("Missing_Count") / total_count * 100)
).orderBy(F.desc("Missing_Count"))

# Afficher les résultats
print("\nValeurs manquantes par colonne sous 'data':")
missing_df_with_percent.show(total_count, truncate=False) 




Nombre total de lignes : 55691
Nombre de lignes où 'data' est null : 0

Valeurs manquantes par colonne sous 'data':
+-----------------+-------------+----------+
|Column           |Missing_Count|Percentage|
+-----------------+-------------+----------+
|discount         |0            |0.0       |
|appid            |0            |0.0       |
|languages        |0            |0.0       |
|categories       |0            |0.0       |
|short_description|0            |0.0       |
|name             |0            |0.0       |
|ccu              |0            |0.0       |
|tags             |0            |0.0       |
|platforms        |0            |0.0       |
|type             |0            |0.0       |
|developer        |0            |0.0       |
|website          |0            |0.0       |
|release_date     |0            |0.0       |
|positive         |0            |0.0       |
|negative         |0            |0.0       |
|price            |0            |0.0       |
|required_age     |0         

In [0]:
display(missing_df_with_percent)

Column,Missing_Count,Percentage
appid,0,0.0
discount,0,0.0
release_date,0,0.0
genre,0,0.0
languages,0,0.0
header_image,0,0.0
required_age,0,0.0
initialprice,0,0.0
platforms,0,0.0
name,0,0.0


In [0]:
#Filtre pour les publishers (on enlève les publishers qui n'ont pas de nom)

print(f"Nombre de lignes avant : {steam.count()}")
steam = steam.filter(F.col("data.publisher").isNotNull() & (F.trim(F.col("data.publisher")) != "") & (F.trim(F.col("data.publisher")) != ".") & (F.trim(F.col("data.publisher")) != "...") & (F.trim(F.col("data.publisher")) != "(none)") )
print(f"Nombre de lignes après : {steam.count()}")

Nombre de lignes avant : 55691
Nombre de lignes après : 55551


In [0]:

# Top 20 des plus grand éditeurs par le nombre de jeux publiés
# Postulat appid = l'id d'un jeu publié

app_published_df = steam.select(
    F.col("data.appid").alias("appid"), 
    F.col("data.publisher").alias("publisher")

).groupBy("publisher").count().orderBy(F.desc("count"))

app_published_df = app_published_df.withColumnRenamed("count", "total_games").limit(20)

display(app_published_df, title="Top 20 éditeurs par nombre de jeux publiés")

publisher,total_games
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
Sekai Project,132
HH-Games,132
Ubisoft,127
Laush Studio,126


Databricks visualization. Run in Databricks to view.

In [0]:
#Top 10 des jeux ayant récolté de plus d'avis positifs

from pyspark.sql import functions as F

# Trouver le jeu avec le plus de points positifs
most_positive_game = steam.select(
    F.col("data.positive").alias("positive_rating"),
    F.col("data.name").alias("game_name")
).orderBy(F.desc("positive_rating")).limit(10)

# Afficher le résultat
display(most_positive_game)

positive_rating,game_name
5943345,Counter-Strike: Global Offensive
1534895,Dota 2
1229265,Grand Theft Auto V
1185361,PUBG: BATTLEGROUNDS
1014711,Terraria
942910,Tom Clancy's Rainbow Six Siege
861240,Garry's Mod
846407,Team Fortress 2
732513,Rust
643836,Left 4 Dead 2


Databricks visualization. Run in Databricks to view.

In [0]:
#Date et nombres de jeux

from pyspark.sql.functions import to_date, col, year


# Correction : Utiliser le bon format "yyyy/M/d"
date_release = steam.select(
    col("data.appid").alias("Appid"),
    to_date(col("data.release_date"), "yyyy/M/d").alias("Release_Date")
).orderBy(col("Release_Date").desc())

date_release.show(truncate=True)

+-------+------------+
|  Appid|Release_Date|
+-------+------------+
|2000040|  2022-11-11|
|1134700|  2022-11-10|
|1287530|  2022-11-10|
| 997010|  2022-11-10|
|1904540|  2022-11-07|
|2184640|  2022-11-07|
| 986040|  2022-11-07|
|1393460|  2022-11-06|
|1928350|  2022-11-06|
|2182020|  2022-11-06|
|2189420|  2022-11-06|
|1450800|  2022-11-05|
|1690650|  2022-11-05|
|1925830|  2022-11-05|
|2142370|  2022-11-05|
|2162150|  2022-11-05|
|2162780|  2022-11-05|
|1622770|  2022-11-04|
|1939560|  2022-11-04|
|1886270|  2022-11-04|
+-------+------------+
only showing top 20 rows



In [0]:
#Affichage des 5 plus sorties de jeux trié par année
# Postulat : la période Covid s'étend entre 2020 et 2021 est la période la plus productive

from pyspark.sql.functions import year, to_date

date_release_format = (
    date_release
    # D'abord convertir la colonne string en date
    .withColumn("Year", year("Release_Date"))
    # Garder les colonnes voulues
    .select("Appid", "Year")
    .groupBy("Year").count().orderBy(F.desc("count")).limit(5)
)

display(date_release_format)

Year,count
2021,8801
2020,8277
2018,7629
2022,7451
2019,6940


Databricks visualization. Run in Databricks to view.

In [0]:
# Correction : Utiliser le bon format "yyyy/M/d"
date_release = steam.select(
    col("data.appid").alias("Appid"),
    to_date(col("data.release_date"), "yyyy/M/d").alias("Release_Date")
).withColumn("Year", year(col("Release_Date"))).withColumn("Month", month(col("Release_Date")))

# Filtrer pour l'année 2022
date_release_2022 = date_release.filter(col("Year") == 2022)

# Regrouper par mois pour l'année 2022
date_release_grouped_2022 = date_release_2022.groupBy("Month").count().orderBy("Month")

display(date_release_grouped_2022)

# L'année 2022 est incomplète, difficile à dire dire si l'engouement depuis le Covid a baissé ou non.


Month,count
1,681
2,652
3,816
4,625
5,771
6,677
7,708
8,759
9,783
10,818


In [0]:
#conversion en float, division par 100 car le prix de départ en cent
steam = steam.withColumn(
    "price_float",
    col("data.price").cast("float")/100
)
#Filtre prix < 200$
prix = steam.select("price_float").groupBy("price_float").count()
prix_distribution = prix.filter(col("price_float") < 200)

In [0]:
from pyspark.sql.functions import sum
prix_sup_70_filter = prix.filter(col("price_float") > 70)

total_count = prix_sup_70_filter.agg(sum("count")).collect()[0][0]

print(f"La somme totale de la colonne count est: {total_count}")

In [0]:
prix_distribution = (
    prix.filter(col("price_float") < 150)
    .orderBy(F.desc('count'))
    .withColumn(
        "Tranche",
        F.when(F.col("price_float") < 10, "en dessous de 10")
         .when((F.col("price_float") >= 10) & (F.col("price_float") < 20), "Entre 10 et 20")
         .when((F.col("price_float") >= 20) & (F.col("price_float") < 30), "Entre 20 et 30")
         .when((F.col("price_float") >= 30) & (F.col("price_float") < 40), "Entre 30 et 40")
         .when((F.col("price_float") >= 40) & (F.col("price_float") < 50), "Entre 40 et 50")
         .otherwise("50 ou plus")  
    )
)

In [0]:
#Distribution prix en dessous de 200$
display(prix_distribution)

In [0]:
# Conversion en Float
steam = steam.withColumn("price_float", col("data.price").cast("float") / 100)

# Création des tranches de prix
prix_distribution = steam.withColumn(
    "Tranche",
    F.when(F.col("price_float") < 10, "en dessous de 10")
     .when((F.col("price_float") >= 10) & (F.col("price_float") < 20), "Entre 10 et 20")
     .when((F.col("price_float") >= 20) & (F.col("price_float") < 30), "Entre 20 et 30")
     .when((F.col("price_float") >= 30) & (F.col("price_float") < 40), "Entre 30 et 40")
     .when((F.col("price_float") >= 40) & (F.col("price_float") < 50), "Entre 40 et 50")
     .otherwise("50 ou plus")
)

# Calcul du chiffre d'affaires par tranche de prix
ca_par_tranche = prix_distribution.groupBy("Tranche").agg(
    F.sum("price_float").alias("Chiffre_d_affaires")
).orderBy(F.desc("Chiffre_d_affaires"))

display(ca_par_tranche)

In [0]:
# Affichage du résultat des discount
#Creation d'une nouvelle colonne pour convertir la colonne discount en integer.
steam = steam.withColumn(
    "discount_int",
    col("data.discount").cast("integer") 
)
is_discount = steam.filter(col("discount_int") > 0)
# Affichage du résultat
total_discount = is_discount.select("discount_int").count()
nombre_jeux_total = steam.count()
#Pourcentage de produit en discount
total_jeux_discount = (total_discount / nombre_jeux_total) * 100
print(f"Nombre de jeux avec discount : {round(total_jeux_discount, 2)}%")




In [0]:
#spark = SparkSession.builder.getOrCreate()
df_discount = {
    "Jeux hors discount": round((100 - total_jeux_discount), 2),
    "Jeux discount": round(total_jeux_discount, 2)
}
# Convertir le dictionnaire en une liste de tuples
data = [(key, value) for key, value in df_discount.items()]
# Définir le schéma
schema = StructType([
    StructField("Categorie", StringType(), nullable=False),
    StructField("Valeur", DoubleType(), nullable=False)
])
#Créer le DataFrame Spark
df_discount_spark = spark.createDataFrame(data, schema=schema)

#Afficher le résultat
display(df_discount_spark)

In [0]:
languages = steam.groupBy('data.languages').count()
# Transform la colonne languages en list
languages = languages.withColumn("languages_list", split("languages", ",\s*"))
# Explode de la colonne
languages_exploded = languages.withColumn("language", explode("languages_list"))

In [0]:
top_language = languages_exploded.groupBy('language').count().orderBy(F.desc("count")).limit(10)
display(top_language)

In [0]:
#Répartition des âges des jeux de Steam
#conversion en Integer pour les âges
steam = steam.withColumn(
    "required_age",
    col("data.required_age").cast("integer") 
)
age = steam.select("data.required_age")

# Renommage des catégories
steam = steam.withColumn(
    "age_category",
     when(col("required_age") == 0, "SANS_RESTRICTION")  
     .when(col("required_age") <= 16, "MINEUR")
     .otherwise("MAJEUR")
)


In [0]:
display(steam.select('age_category'))

In [0]:
#Afficher le top des Genre (limite à 10)

display(steam.groupBy('data.genre').count().orderBy(F.desc("count")).limit(10))

In [0]:
# Filtre correct avec les ratings supérieurs à 5000 (pour réduire la voilure)
filtered_reviews = steam.filter(
    (col("data.positive") > 5000) & (col("data.negative") > 5000)
)

# Ratio entre Positive/Negative
genre_stats = filtered_reviews.select(
    col("data.genre").alias("Genre"),
    col("data.positive").alias("Positive_rating"),
    col("data.negative").alias("Negative_rating")
).groupBy("Genre").agg(
    F.count("*").alias("Game_Count"),
    F.sum("Positive_rating").alias("Total_Positive"),
    F.sum("Negative_rating").alias("Total_Negative"),
    (F.sum("Positive_rating") / F.sum("Negative_rating")).alias("Positive_Negative_Ratio")
).orderBy(F.desc("Positive_Negative_Ratio")).limit(20)


# Affichage des résultats avec formatage
genre_ratio = genre_stats.withColumn(
    "Positive_Negative_Ratio", 
    F.round(F.col("Positive_Negative_Ratio"), 2)
)

display(genre_ratio)

In [0]:
display(genre_ratio)

In [0]:
# Top 20 Publishers par Genre
genre_publisher = steam.select(
    F.col("data.genre").alias("Genre"),
    F.col("data.publisher").alias("Publisher"),
    
).groupby('publisher', 'genre').count().orderBy(F.desc("count")).limit(20)

display(genre_publisher)



In [0]:
#Multigenre : Classement des publishers qui publient dans plusieurs genres différents
from pyspark.sql import functions as F

publisher_genre_stats = steam.select(
    F.col("data.publisher").alias("Publisher"),
    F.col("data.genre").alias("Genre")
).groupBy("Publisher").agg(
    F.countDistinct("Genre").alias("Nombre_de_genres_distincts"),
    F.collect_set("Genre").alias("Liste_des_genres")
).filter("Nombre_de_genres_distincts > 1").orderBy(
    F.desc("Nombre_de_genres_distincts")
)

display(publisher_genre_stats)

In [0]:
from pyspark.sql import functions as F

publisher_genre_stats = steam.select(
    F.col("data.publisher").alias("Publisher"),
    F.col("data.genre").alias("Genre")
).groupBy("Publisher").agg(
    F.countDistinct("Genre").alias("Nombre_de_genres_distincts"),
    F.collect_set("Genre").alias("Liste_des_genres")
).filter("Nombre_de_genres_distincts > 1").orderBy(
    F.desc("Nombre_de_genres_distincts")
).limit(10)

display(publisher_genre_stats)



In [0]:
#Genre de prédilection : classement des publishers qui ont peu de genre vers ceux qui couvrent le plus de genre

publisher_genre_stats = steam.select(
    F.col("data.publisher").alias("Publisher"),
    F.col("data.genre").alias("Genre")
).groupBy("Publisher").agg(
    F.countDistinct("Genre").alias("Nombre_de_genres_distincts"),
    F.collect_set("Genre").alias("Liste_des_genres")
).filter("Nombre_de_genres_distincts > 1").orderBy(
    F.asc("Nombre_de_genres_distincts")
)

display(publisher_genre_stats)

In [0]:
## Question 4: What are the most lucrative genres?

publisher_prix = steam.select(
    F.col("data.genre").alias("Genre"),
    F.col("data.price").alias("Prix")
).groupBy("Genre").agg(F.sum("Prix").alias("Revenu_total")).orderBy(F.desc("Revenu_total")).limit(20)

display(publisher_prix)

In [0]:
plateformes = steam.groupBy("data.platforms").count().orderBy(F.desc("Count"))
display(plateformes)


In [0]:
plateformes = steam.groupBy("data.platforms").count().orderBy(F.desc("Count"))
plateformes.show()



In [0]:

#Répartition des Plateformes de jeux disponibles

from pyspark.sql import functions as F

df_plateformes = plateformes.withColumn(
    "platforms_str",
    F.when(
        (~F.col("platforms.windows") & ~F.col("platforms.mac") & F.col("platforms.linux")), "LINUX"
    ).when(
        (F.col("platforms.windows") & F.col("platforms.mac") & F.col("platforms.linux")), "WINDOWS/MAC/LINUX"
    ).when(
        (~F.col("platforms.windows") & F.col("platforms.mac") & F.col("platforms.linux")), "MAC/LINUX"
    ).when(
        (F.col("platforms.windows") & F.col("platforms.mac") & ~F.col("platforms.linux")), "WINDOWS/MAC"
    ).when(
        (F.col("platforms.windows") & ~F.col("platforms.mac") & F.col("platforms.linux")), "WINDOWS/LINUX"
    ).when(
        (~F.col("platforms.windows") & F.col("platforms.mac") & ~F.col("platforms.linux")), "MAC"
    ).when(
        (F.col("platforms.windows") & ~F.col("platforms.mac") & ~F.col("platforms.linux")), "WINDOWS"
    ).otherwise("OTHER")  # Cas non gérés
)

display(df_plateformes)

In [0]:
# Sélection des données de base
genre_platforms = steam.select(
    F.col("data.genre").alias("Genre"),
    F.col("data.platforms").alias("Plateformes"),
    
).groupby('Plateformes', 'Genre').count().orderBy(F.desc("count"))

display(genre_platforms)

In [0]:
df_genre_plateformes = genre_platforms.withColumn(
    "platforms_str",
    F.when(
        (~F.col("Plateformes.windows") & ~F.col("Plateformes.mac") & F.col("Plateformes.linux")), "LINUX"
    ).when(
        (F.col("Plateformes.windows") & F.col("Plateformes.mac") & F.col("Plateformes.linux")), "WINDOWS/MAC/LINUX"
    ).when(
        (~F.col("Plateformes.windows") & F.col("Plateformes.mac") & F.col("Plateformes.linux")), "MAC/LINUX"
    ).when(
        (F.col("Plateformes.windows") & F.col("Plateformes.mac") & ~F.col("Plateformes.linux")), "WINDOWS/MAC"
    ).when(
        (F.col("Plateformes.windows") & ~F.col("Plateformes.mac") & F.col("Plateformes.linux")), "WINDOWS/LINUX"
    ).when(
        (~F.col("Plateformes.windows") & F.col("Plateformes.mac") & ~F.col("Plateformes.linux")), "MAC"
    ).when(
        (F.col("Plateformes.windows") & ~F.col("Plateformes.mac") & ~F.col("Plateformes.linux")), "WINDOWS"
    ).otherwise("OTHER") 
)

display(df_genre_plateformes)

In [0]:
from pyspark.sql import functions as F

genre_by_platforms_agg = df_genre_plateformes.groupby('platforms_str') \
    .agg(
        F.collect_set('Genre').alias('Genres_uniques'),  # collect_set supprime les doublons
        F.count('*').alias('Nombre_de_jeux')
    ) \
    .orderBy(F.desc('Nombre_de_jeux')) \
    .withColumnRenamed('platforms_str', 'Plateformes')

display(genre_by_platforms_agg)

In [0]:
from pyspark.sql import functions as F


genre_by_platforms_agg = df_genre_plateformes.groupby('platforms_str') \
    .agg(
        F.collect_set('Genre').alias('Genres_uniques_liste'),  # Évite les doublons
        F.count('*').alias('Nombre_de_jeux')
    ) \
    .withColumnRenamed('platforms_str', 'Plateformes')

genre_by_platforms_merged = genre_by_platforms_agg.withColumn(
    "Genres_uniques",  
    F.concat_ws(", ", F.col("Genres_uniques_liste")) 
).drop("Genres_uniques_liste")  

#Ordonner par nombre de jeux décroissant
final_result = genre_by_platforms_merged.orderBy(F.desc("Nombre_de_jeux"))

display(final_result)

In [0]:
#Répartition des genres selon les plateformes et classement

from pyspark.sql.types import StringType
import re


def dedup_genres(genres_str):
    if not genres_str:
        return genres_str
    genres_list = genres_str.split(", ")
    unique_genres = list(set(genres_list))
    unique_genres = ", ".join(sorted(unique_genres))
    unique_genres = re.sub(r'^,', '', unique_genres)
    return unique_genres

dedup_genres_udf = F.udf(dedup_genres, StringType())

df_final = genre_by_platforms_merged.withColumn(
    "Genres_uniques_dedup",
    dedup_genres_udf(F.col("Genres_uniques"))
)

df_final = df_final.drop("Genres_uniques")

df_final = df_final.orderBy(F.col("nombre_de_jeux").desc())

display(df_final)
